### Functions/code order:
1. `2-ukb_target_pipe`
* Generate targets and select features by utility.
2. `Link_semmed_cuis`
* NER linking + KG linkage/graph filtering. 
* Also contains code for future DB validation/checking (what appears in future). 
* By default, output only cases with 0 kg hits. 
* NOTE! We want to keep also the feature names from this.
* Also does filtering by semantic sim, and removes features with near exact match in kg (i.e if a feature's top entity is almost identical to the feature, and that entity has a kg match, then drop feature and "children" concepts). 
* 
Outputs concepts/UMLS entities associated with features. (And 0 KG hits) .
3. `search_pubmed`
* Literature search. Also by concept/entity. 
* +- Filter by KG distance, i.e distance 2 (1 hop between target and entity). 

4. `run_pipe-llmCall.ipynb` : LLM - run med_Rag and rank outputs. 
* Reformats feature + disease as prompt(s), ) 
* - can run local or api model. and runs rag then sends to LLM using `MedRAG` (with modified prompt template, code)
* Note - needs extracting into terms and doing RAG, +- custom search. **Slow. **


5. (Here) - join picked features with feature feats (metadata) for final filtering; and run additional filter/rank+ on outputs, maybe without rag context
--------------------------------

## Warning! Delete before upload!!!!
* IMPORTANT!!

In [ ]:
import os
import openai
# Initialize OpenAI API (ensure you have set your API key securely)
openai.api_key = os.getenv("OPENAI_API_KEY") 

In [ ]:
import warnings
warnings.filterwarnings("ignore")
# warnings.filterwarnings("W036")
import logging
logger = logging.getLogger("spacy")
logger.setLevel(logging.ERROR)
import pandas as pd
import json
import re
from tqdm import tqdm

import pandas as pd

import json
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List
import time

from ukb_target_pipe  import  make_target_df,model_features,ipw_downsampling#*
from search_pubmed import  run_search_pubmed #*
from configs import * #config_gall
from Link_semmed_cuis import  * #link_kg_concepts #*
from util import generate_boring_prompts, deduplicate_texts

In [ ]:
import openai
from pydantic import BaseModel
import json
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List
import time
import re
import logging

# Configure logging
logging.basicConfig(
    level=logging.WARNING #logging.DEBUG,  # Change to INFO or WARNING to reduce verbosity
    ,format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("feature_evaluation.log"),
        logging.StreamHandler()
    ]
)

# Define Pydantic Models
class FeatureEvaluation(BaseModel):
    step_by_step_explanation: str = Field(..., description="Detailed reasoning for the evaluation based on the criteria.")
    answer: bool = Field(..., description="Boolean indicating if the feature is interesting (True) or not (False).")
    numeric_score: int = Field(..., ge=1, le=4, description="Numeric score between 1 and 4 representing the level of interest/novelty.")

class FeatureEvaluationWithError(BaseModel):
    step_by_step_explanation: Optional[str] = Field(None, description="Detailed reasoning for the evaluation based on the criteria.")
    answer: Optional[bool] = Field(None, description="Boolean indicating if the feature is interesting (True) or not (False).")
    numeric_score: Optional[int] = Field(None, ge=1, le=4, description="Numeric score between 1 and 4 representing the level of interest/novelty.")
    error: Optional[str] = Field(None, description="Error message if parsing or validation fails.")

def extract_json_from_response(response: str) -> Optional[str]:
    """
    Extract JSON content from a response string enclosed in code blocks.
    """
    json_pattern = re.compile(r'```json\s*\n?(.*?)\n?```', re.DOTALL | re.IGNORECASE)
    match = json_pattern.search(response)
    if match:
        json_str = match.group(1).strip()
        logging.debug(f"Extracted JSON: {json_str}")
        return json_str
    else:
        # Attempt to extract JSON without code blocks
        start = response.find('{')
        end = response.rfind('}')
        if start != -1 and end != -1 and end > start:
            json_str = response[start:end+1].strip()
            logging.debug(f"Extracted JSON without code blocks: {json_str}")
            return json_str
    logging.warning("Failed to extract JSON from the response.")
    return None

def evaluate_features(prompts: List[dict], sleep_time: float = 1.1, max_retries: int = 1,model_name="gpt-4o-mini") -> pd.DataFrame:
    evaluations = []
    for idx, item in enumerate(prompts):
        feature = item['feature']
        target = item['target']
        prompt = item['prompt']
        attempt = 0
        success = False
        
        while attempt < max_retries and not success:
            try:
                logging.info(f"Evaluating feature '{feature}'. Attempt {attempt + 1}/{max_retries}.")
                response = openai.ChatCompletion.create(
                    model=model_name,
                    messages=[
                        {"role": "system", "content": "You are an clever research assistant that evaluates candidate features based on novelty, plausibility, and usefulness."},
                        {"role": "user", "content": prompt}
                    ],
                    max_completion_tokens=700,  # Ensure sufficient tokens for JSON
                    temperature=0.3,  # Deterministic output
                    n=1,  # Single response
                    # stop=None,
                    ## add:
                    # response_format = FeatureEvaluation
                    response_format={
                   'type': 'json_schema',
                   'json_schema': 
                      {
                        "name":"whocares", 
                        "schema": FeatureEvaluation.model_json_schema()
                      }
                 } 
                )
                
                # # Extract the assistant's reply
                # print("response.choices[0]")
                # print(response.choices[0].message['content'])
                reply = response.choices[0].message['content'].strip()
                # reply = response.choices[0].message.parsed
                
                logging.debug(f"Raw GPT-4 response for feature '{feature}': {reply}")
                
                # # Extract JSON from the response
                # extracted_json = extract_json_from_response(reply)
                extracted_json =reply 
                
                if not extracted_json:
                    raise ValueError("Failed to extract JSON from the response.")
                
                # Parse the JSON content
                try:
                    evaluation_data = json.loads(extracted_json)
                    evaluation = FeatureEvaluation(**evaluation_data)
                    evaluations.append({
                        'feature': feature,
                        'target': target,
                        'step_by_step_explanation': evaluation.step_by_step_explanation,
                        'answer': evaluation.answer,
                        'numeric_score': evaluation.numeric_score,
                        'error': None
                    })
                    logging.info(f"Successfully evaluated feature '{feature}'.")
                    success = True  # Exit the retry loop
                except (json.JSONDecodeError, ValidationError) as e:
                    raise ValueError(f"Parsing/Validation Error: {str(e)}. Extracted JSON: {extracted_json}")
            
            except (openai.error.OpenAIError, ValueError) as e:
                attempt += 1
                logging.error(f"Error evaluating feature '{feature}': {str(e)}")
                if attempt < max_retries:
                    logging.info(f"Retrying feature '{feature}' after error.")
                    time.sleep(sleep_time)  # Wait before retrying
                else:
                    # Record the error after exhausting retries
                    evaluations.append({
                        'feature': feature,
                        'target': target,
                        'step_by_step_explanation': None,
                        'answer': None,
                        'numeric_score': None,
                        'error': str(e)
                    })
        
        # Optional: Sleep to respect rate limits between features
        time.sleep(sleep_time)
    
    # Convert the evaluations list to a DataFrame
    evaluations_df = pd.DataFrame(evaluations)
    
    return evaluations_df

def generate_extra_prompt(data: pd.DataFrame) -> List[dict]:
    results = []
    for index, row in data.iterrows():
        # Clean and prepare data
        feature_name_clean = row['feature']
        
        target = row['target']
        novelty_cot = row['novel_cot']
        plausible_cot = row['plausible_cot']
        # utility = row['utility']
        corr = row['corr']
        # p_val = row['p_val']
        feature_split = row['F.Split-Feature Split']
        
        # Determine direction of effect
        if corr > 0:
            direction = 'positive'
        elif corr < 0:
            direction = 'negative'
        else:
            direction = 'neutral'
        
        # Construct the prompt with an example response
        prompt = (
            f"Evaluate the feature '{row['raw_name']}' in relation to predicting the target disease: '{target}'. The feature has a {direction} correlation with the target disease (when predicting 1 year in advance, and after controlling for age, gender and BMI; so magnitude of correlation or feature importance are less important).\n\n"
            f"### Criteria Definitions:\n"
             f"- **Novelty:** Assess whether the feature ({feature_name_clean}) provides new insights, contradicts established understanding, or explores controversial associations not well-documented in existing literature. (i.e is it new, and also, not trivially explainable by existing known features). \n"
            f"- **Plausibility:** Evaluate if the association makes logical sense based on known mechanisms, biological pathways, social or environmental factors or established risk factors.\n"
            f"- **Usefulness/utility:** (Optional) Does the feature have any potential practical applications or utility, such as informing clinical interventions or tests, detection, usage in models or policy implications.\n\n"
            f"### Existing Explanations:\n These explanations are from weak critics and some literature, so you may regard them at your discretion or rely on your own knowledge and step by step analysis.\n"
            f"**Novelty Explanation:**\n{novelty_cot}\n\n"
            f"**Plausibility Explanation:**\n{plausible_cot}\n\n"
            f"### Additional Information:\n"
            # f"- **Utility Score:** {utility}\n"
            # f"- **Correlation:** {corr}\n"
            # f"- **p-value:** {p_val}\n"
            f"- **Feature Split:** {feature_split}\n"
            f"- **Feature Lift (for target==True) under feature split:** {row['F.Split-Lift (y==1)']}\n"
            f"Evaluate how **interesting** this feature is to a medical researcher, biologist, clinician or basic research. Take into account world knowledge, analysis, vibes and also the criteria of **novelty** and **plausibility**.\n\n"
            f"**Instructions:**\n"
            f"1. **Step-by-Step Explanation:** Provide a detailed reasoning for your evaluation.\n"
            f"2. **Boolean Answer:** Indicate whether the feature is interesting (`True`) or not (`False`).\n"
            f"3. **Numeric Score:** Assign a score between 1 and 5, where 1 = \"Not interesting/novel at all\" and 4 = \"Extremely novel and interesting\". \n\n"
            f"**Output Format:**\n"
            f"Provide your response in **JSON format** strictly adhering to the schema:\n"
            f"```json\n"
            f"{{\n"
            f"  \"step_by_step_explanation\": \"<Your detailed explanation>\",\n"
            f"  \"answer\": <True/False>,\n"
            f"  \"numeric_score\": <1-4>\n"
            f"}}\n"
            f"```\n\n"
            f"**Example Response:**\n"
            f"```json\n"
            f"{{\n"
            f"  \"step_by_step_explanation\": \"The feature 'X' shows a significant association with lower 'Y', this is opposite to the directions expected from known literature or science, as X would be expected to have an opposite effect due to its involvement in Z.\",\n"
            f"  \"answer\": True,\n"
            f"  \"numeric_score\": 4\n"
            f"}}\n"
            f"```\n\n"
            f"**Ensure that the JSON is valid and follows the exact structure without any additional fields or deviations. Do not include any text outside of the JSON block.**"
        )
        
        # Store the prompts
        results.append({
            'feature': row['feature'],
            'target': row['target'],
            'prompt': prompt
        })
        
    return results


In [ ]:
# Fast_Run = True
Fast_Run = False
# SAVE_OUTPUT = True

# RUN_PIPE = True
SAVE_OUTPUT = False

run_LLM_rerank = False ## Set to true to run the gpt 4 - costs money!!
run_get_boring = False # uses open llm, add boring annot used in negatives. similar but not identical to "novel" filter!

# OUTPUT_NAME= "localMini_fast.csv"
# OUTPUT_NAME="4_mini.csv"
# OUTPUT_NAME="4_full.csv"
OUTPUT_NAME = "extra.csv"

In [ ]:
from configs import *
## import from configs
all_configs = [
        config_celiac, config_gall,config_gout, 
    config_spine, config_oesophagus,
    config_heart, config_eye_occ,config_depression
              ]
if Fast_Run:
    all_configs = [config_eye_occ]
config = all_configs[0]

In [ ]:
import sys
### orig
# # sys.path.append(r'D:\Research2\MedRAG')
# # sys.path.append(r'D:\Research2\MedRAG\src')
# sys.path.append('/mnt/d/Research2/MedRAG/')
# sys.path.append('/mnt/d/Research2/MedRAG/src')

# sys.path.append('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag')

# ## change to path with medrag and it's downloaded corpus!
# # os.chdir('/mnt/d/Research2/MedRAG/')
# os.chdir('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag')
# # from src.medrag import MedRAG
##########
## new: 
sys.path.append('/mnt/d/Research2/MedRAG/')
sys.path.append('/mnt/d/Research2/MedRAG/src')

sys.path.append('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/')
sys.path.append('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/src')
sys.path.append('/mnt/d/Research2/MedRAG/')
## change to path with medrag and it's downloaded corpus!
# os.chdir('/mnt/d/Research2/MedRAG/')
# os.chdir('/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag')
from src.medrag import MedRAG
from tqdm import tqdm
import json

from util import get_predictions_from_medrag

# CORPUS_DIR_CACHE_PATH = '/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/corpus'
CORPUS_DIR_CACHE_PATH = '/mnt/d/Research2/MedRAG/corpus'

## Use additionalLLM for ranking of candidates
* 

In [ ]:
for config in all_configs:
    print("---"*40,"\n",config['OUTPUT_RES_PREFIX'],"\n")
    config["df"] = pd.read_csv(f"{config.get('OUTPUT_RES_PREFIX', '')}{config.get('filtered_results_filename', 'review_interesting_candidates_results.csv')}")
    print(config['OUTPUT_RES_PREFIX']+OUTPUT_NAME)
    print(config["df"].shape)

In [ ]:
%%time
if run_LLM_rerank:
    for config in all_configs:
        print("---"*40,"\n",config['OUTPUT_RES_PREFIX'],"\n")
        ## "./Outputs/llminterestingfeaturesoutputs"+config['OUTPUT_RES_PREFIX']+"_gpt4.csv" # - local output - but!:
        ## we are in the medrag folder currently! 
        
        df_llm_rated = pd.read_csv(config['OUTPUT_RES_PREFIX']+"4_mini.csv")
        print(df_llm_rated.shape,"df_llm_rated")
        print(df_llm_rated.select_dtypes(exclude=["O"]).mean().round(4))
        
        df_merged = df_llm_rated.merge(config["df"],left_on="feature",right_on="feature_name",how="inner")
        assert df_merged.shape[0]==config["df"].shape[0]
        
        df_candidates = df_merged.loc[df_merged[['novel','plausible']].max(axis=1)>0].dropna(axis=1,how="all").drop_duplicates()
        df_candidates.sort_values(['novel','plausible', 'utility', 'MutualInfoTarget','shortest_path_length', 'feature_importance',],ascending=False,inplace=True)
        df_candidates = df_candidates.loc[(df_candidates["feature_importance"]>=0.001)|(df_candidates["MutualInfoTarget"]>=0.001)]
        
        df_candidates = df_candidates[['feature', 
               'novel_cot', 'plausible_cot', 'novel', 'plausible', 'utility',
                                       # 'feature_name',
                                       'shortest_path_length',
                                       'corr', 'raw_name',
                                       'p_val', 
               'feature_importance', 'MutualInfoTarget',
                                       'F.Split-Lift (y==1)', 'F.Split-Feature Split', 'Target']].reset_index(drop=True)
        display(df_candidates.drop(columns=['novel', 'plausible', 'utility'],axis=1).round(2).head(2))
        # df_candidates.head().to_csv("sample_candidates_fullMetadata.csv",index=False)
    
        
        df = df_candidates.rename(columns={"Target":"target"},errors="ignore")
        if Fast_Run:
            df = df.sample(2)
        # Generate prompts
        prompts = generate_extra_prompt(df)
        
        # Evaluate features
        evaluations_df = evaluate_features(prompts,model_name= 'gpt-4o-mini' if Fast_Run else 'gpt-4o' )
        
        # Merge the evaluations with the original dataframe
        final_df = df.merge(evaluations_df, left_on=['feature', 'target'], right_on=['feature', 'target'], how='left')
        final_df.dropna(axis=1,how="all",inplace=True)
        try:
            final_df["answer"] = pd.to_numeric(final_df["answer"])
        except:()
        try:
            final_df = final_df.sort_values(['novel','plausible', "answer","numeric_score",'utility', 'MutualInfoTarget','shortest_path_length', 'feature_importance',],ascending=False)
        except:()
        # Display the final dataframe
        display(final_df)
        if SAVE_OUTPUT:
            final_df.to_csv(config['OUTPUT_RES_PREFIX']+"_reranked_gpt4.csv",index=False)
    

#### add extra "boring" (non novel) from extra small model with rag - redundnat , but added for some extra ranking/comparisons
* NOT USED in end!
* Overwrites orig files

In [ ]:
%%time
if run_get_boring:
    medrag = MedRAG(llm_name="OpenScholar/Llama-3.1_OpenScholar-8B", rag=True,
                # retriever_name="BM25" # "BM25" #"RRF-2"#"MedCPT" - RRF2 woudl take ~ 11 hours for pubmed
                retriever_name="RRF-2" ## fast enough with small data
                , corpus_name="MedText"  ## small 
                ,HNSW=True 
                ,corpus_cache=True
                ,db_dir= CORPUS_DIR_CACHE_PATH#'/Users/oferd2/Library/CloudStorage/OneDrive-MedtronicPLC/Documents/research/MedRag/corpus',
               )
    for config in all_configs:
        print("--"*30,f"\n {config['OUTPUT_RES_PREFIX']}")
        df = pd.read_csv("./Outputs/llm_reranked/"+config['OUTPUT_RES_PREFIX']+"_reranked_gpt4.csv")
        # df_raw = df.copy()
        ## sample
        if Fast_Run:
            df = df.query('answer & (novel>0) & (numeric_score>=4)').head(8).copy()
            # df = df.head(3).copy()
        df["feature_name"] = df["feature"]
        df["Target"] = df["target"]
        
        print("medrag boring LLM")
        boring_prompts = generate_boring_prompts(df)
        boring_pred,boring_cot = get_predictions_from_medrag(medrag, boring_prompts, 'boring_prompt', 'boring_options',K=20)
        df["boring_pred"] = boring_pred
        print(df["boring_pred"].value_counts(normalize=True).round(2))
        df["boring_cot"] = boring_cot
        df["boring_pred"] = df["boring_pred"]=="A"
    
        df = df.drop(columns=["feature_name","Target"],errors="ignore")
        # 
        if SAVE_OUTPUT:
            if not Fast_Run:
                df.to_csv("./Outputs/llm_reranked/"+config['OUTPUT_RES_PREFIX']+"_reranked_gpt4.csv",index=False)
        
        # df#.head()
        df_bore = df.query('boring_pred & (novel>0) & (numeric_score>=4)')[['feature',  'numeric_score',"step_by_step_explanation",'boring_cot', 'boring_pred',"novel"]]
        print("boring cases vs contradicting explanation:")
        # display(df_bore[["boring_cot","step_by_step_explanation"]].head())
        print(df_bore[["boring_cot","step_by_step_explanation"]].head(3).values)

In [ ]:
print(config['OUTPUT_RES_PREFIX'])
df = pd.read_csv("./Outputs/llm_reranked/"+config['OUTPUT_RES_PREFIX']+"_reranked_gpt4.csv")
df

In [ ]:
df_bore = df.query('boring_pred & (novel>0) & (numeric_score>=4)')[['feature',  'numeric_score',"step_by_step_explanation",'boring_cot', 'boring_pred',"novel"]]
print("boring cases vs contradicting explanation:")
# display(df_bore[["boring_cot","step_by_step_explanation"]].head())
print(df_bore[["boring_cot","step_by_step_explanation"]].head(3).values)

In [ ]:
# df_bore = df.query('boring_pred')[['feature',  'numeric_score',"step_by_step_explanation",'boring_cot', 'boring_pred',"novel"]]
# df_not_bore = df.query('~boring_pred')[['feature',  'numeric_score','boring_cot', 'boring_pred']]
# print("boring cases")
# display(df_bore)

# print(df_bore["boring_cot"].values)

# print("non boring")
# print(df_not_bore.shape[0])
# # print(df_not_bore["boring_cot"].values)

### Output slightly cleaner outputs (final version - use just head amount
* what about p-value filter??

In [ ]:
for config in all_configs:
    print(config['OUTPUT_RES_PREFIX'])
    # result = glob.glob('*.csv')
    x = "./Outputs/llm_reranked/"+config['OUTPUT_RES_PREFIX']+"_reranked_gpt4.csv"
    df_in = pd.read_csv(x)
    df_in.dropna(axis=1,how="all",inplace=True)
    
    df_in["not_boring"] = df_in['boring_pred']==False
    df = df_in[[ 'raw_name', 'novel', 'novel_cot','plausible',  'plausible_cot', 'answer',
           'numeric_score','step_by_step_explanation', 'corr', 'p_val',
           'feature_importance', 'MutualInfoTarget', 'F.Split-Lift (y==1)',
           'F.Split-Feature Split',  "not_boring" ,'boring_cot','target',]].drop_duplicates()
    print(df.shape[0],"# candidates with llm filter filter")
    df = df.loc[(df['answer'])| (df[['novel','plausible']].min(axis=1)>0)]
    # df = df.loc[(df['feature_importance']>=0.001)| ((df['MutualInfoTarget']>=0.01) & (df['p_val']<0.35))]
    print(df.shape[0],"# candidates pre min utility filter")
    ### OPT: Drop features with "missing"! (Not the "None" though")

    # df = df.loc[df['p_val']<0.8] # debug flag
    df = df.loc[(df['p_val']<=0.2)| ((df['MutualInfoTarget']>0.001) & (df['feature_importance']>0.001))]
    print(df.shape[0],"# pre missing/none clean")
    ## harsher filter out of missing/annoyng feat - optional (final pipe doesn't enable them in default due to hardness of reading
    df = df.loc[(~df['raw_name'].str.contains("missing_|None",regex=True)) | (df['p_val']<0.04)]
    
    print(df.shape[0],"# pre dedup")
    clean_feats = deduplicate_texts(df["raw_name"], use_difflib=False,  distance_threshold=0.95) # string_cutoff=0.95,
    print("Dropping:\n",df.loc[~df["raw_name"].isin(clean_feats)]["raw_name"])
    df = df.loc[df["raw_name"].isin(clean_feats)]
    print(df.shape[0],"# candidates")
    
    df.sort_values(["answer", 'numeric_score','novel','plausible', "not_boring" ,'feature_importance','MutualInfoTarget',],ascending=False,inplace=True)
    df.rename(columns={'raw_name':"Feature_Name",'answer':"Interesting?","numeric_score":"Confidence"},inplace=True)
    if SAVE_OUTPUT:
        df.to_csv(f"./Outputs/llm_reranked/subset/"+config['OUTPUT_RES_PREFIX']+"_ranked.csv",index=False)
    

In [ ]:
print(df_bore.query('(novel>0) & (numeric_score>=4)')[["boring_cot","step_by_step_explanation"]].head().values)

In [ ]:
from src.medrag import MedRAG
import pandas as pd
from tqdm import tqdm
import re
from sklearn.metrics import classification_report, roc_auc_score

import json

## REview if some features occur many times
### output figure

In [ ]:
df_concat = None
for config in all_configs:
    # df = pd.read_csv(config['OUTPUT_RES_PREFIX']+"_reranked_gpt4.csv").dropna(axis=1,how="all")
    df = pd.read_csv(os.path.join("Outputs/llm_reranked/",config['OUTPUT_RES_PREFIX']+"_reranked_gpt4.csv")).dropna(axis=1,how="all")
    df = df.drop_duplicates("raw_name").query("p_val<0.4")
    if df_concat is None: # was   if df is None: - probably bug? 
        df_concat = df
    df_concat = pd.concat([df_concat,df ],ignore_index=True)
print(df_concat.shape)
display(df_concat.feature.value_counts())

In [ ]:
## "interesting ranked cases":
df_concat_int = df_concat.query("answer | ((plausible>0) & (novel>0))")
print(df_concat_int.shape[0],"# interesting candidates")
display(df_concat_int.feature.value_counts())

In [ ]:
df_concat.describe().round(2)

In [ ]:
df_concat_int.select_dtypes("O").nunique()

In [ ]:
df_concat.feature.value_counts().hist()

In [ ]:
df_concat_int.feature.value_counts().hist()

In [ ]:
df

In [ ]:
df_concat.feature.value_counts().hist()

## Output ultra final candidates annotations list
* keep subset of columns, and limited # results.
* Should _shuffle_ before sending to each person? 

In [ ]:
MAX_FINAL_CANDIDATES = 50

In [ ]:
print(df.shape[0])
df["answer"] = pd.to_numeric(df["answer"],downcast="integer")
### maybe include plausible? too harsh?
num_pos_candidates = df.loc[df[["novel","answer"]].max(axis=1)!=0].shape[0] 
print(num_pos_candidates,"minimal num_pos_candidates")
df = df.sort_values(["numeric_score","answer",'novel', 'plausible',
                                'utility', 
                                'MutualInfoTarget','shortest_path_length', 'feature_importance',],ascending=False)
# df = df.head(min(num_pos_candidates,MAX_FINAL_CANDIDATES))
# df_can
display(df.round(3))